
# 📘 Day 7 学习笔记：LoRA / QLoRA + 最小 RAG-Agent 闭环

> 学习主线：**全量微调为什么贵 → LoRA 低秩更新 → QLoRA 4-bit 冻结基座 + LoRA → 参数/显存计算 → 最小 RAG Chain → 最小 LangGraph 风格 StateGraph**

---

## 🎯 今日学习大纲

今天是 7 天路线的收束日，目标不是再学一个孤立概念，而是把底层矩阵、显存、推理和应用工程串起来。

### Part A：LoRA / QLoRA

你要解决这些问题：

- 为什么 full fine-tuning 训练成本高？
- LoRA 为什么可以只训练两个小矩阵？
- \(\Delta W = BA\) 的矩阵维度到底是什么？
- LoRA rank \(r\) 如何影响参数量、表达能力和过拟合？
- LoRA 为什么经常挂在 `q_proj / k_proj / v_proj / o_proj / up_proj / down_proj / gate_proj`？
- QLoRA 和 LoRA 的区别是什么？
- NF4、double quantization、paged optimizer 的目的是什么？
- QLoRA 为什么省显存，但 activation 仍然可能 OOM？

### Part B：LangChain / LangGraph 最小闭环

你要解决这些问题：

- RAG Chain 的固定流程是什么？
- Agentic RAG 为什么比普通 RAG 更灵活也更危险？
- LangChain 的 Runnable / LCEL 直觉是什么？
- LangGraph 的 StateGraph 为什么适合复杂 Agent？
- 一个最小 RAG-Agent 应该记录哪些 trace？
- Retriever / Answer / Verify 节点如何拆成状态机？

---

## ✅ 今日完成目标

学完这份 Notebook，你应该能做到：

1. 手写 LoRA 的核心公式：

\[
y = Wx + \frac{\alpha}{r}BAx
\]

2. 精确计算 full fine-tuning 与 LoRA 的参数量差异。
3. 用 PyTorch 写一个最小 LoRA Linear，并只训练 LoRA 参数。
4. 解释 QLoRA 的 4-bit base weights + LoRA adapter + optimizer states 关系。
5. 写一个最小本地 RAG retriever。
6. 写一个 LangGraph 风格的最小状态机：

```text
START → retrieve_context → answer_with_context → verify_answer → END
```

7. 说清楚 RAG / Agent 的 trace 应该记录什么。

---

## 🧠 今日核心概念速览

| 概念 | 一句话解释 |
|---|---|
| Full Fine-tuning | 直接更新原模型全部参数 |
| LoRA | 冻结原权重，只训练低秩增量矩阵 |
| Rank \(r\) | 控制低秩更新子空间大小 |
| Scaling \(\alpha/r\) | 控制 LoRA 更新幅度 |
| QLoRA | 4-bit 量化冻结基座 + 训练 LoRA |
| NF4 | 面向近似正态权重分布的 4-bit 表示 |
| Double Quantization | 再量化量化常数，进一步省显存 |
| Paged Optimizer | 缓解训练时 optimizer state 显存峰值 |
| RAG Chain | 检索 → 拼上下文 → 回答 |
| Agentic RAG | 模型/策略决定是否检索、检索什么、是否验证 |
| StateGraph | 以状态为中心的多节点流程编排 |

---

## 🧭 建议学习顺序

1. 先看 LoRA 数学公式和参数量计算。
2. 运行 LoRA toy training demo，观察 frozen base + trainable adapter。
3. 看 QLoRA 显存计算，理解 4-bit 省在哪里、不省哪里。
4. 跑最小 RAG demo。
5. 跑最小 StateGraph demo。
6. 最后整理一页“面试问答 + 工程排障”。


In [ ]:

# Day 7 所需基础库
import math
import time
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.8)
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(42)
random.seed(42)

print("基础环境就绪 ✅")

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    print("PyTorch 可用 ✅", torch.__version__)
except Exception as e:
    print("PyTorch 未安装。LoRA 训练 demo 需要 torch。")
    print("安装命令：python -m pip install torch")
    raise e



---

# 1️⃣ 为什么 Full Fine-tuning 显存和参数成本高？

假设一个线性层权重是：

\[
W \in \mathbb{R}^{d_{out} \times d_{in}}
\]

标准前向传播：

\[
y = Wx
\]

如果做 full fine-tuning，就意味着直接更新 \(W\)。

---

## 1.1 参数量

\[
\#W = d_{out} \times d_{in}
\]

例如：

\[
d_{out} = 4096, \quad d_{in} = 4096
\]

则：

\[
\#W = 4096 \times 4096 = 16,777,216
\]

单层就约 1,677 万参数。

---

## 1.2 训练显存为什么更贵？

训练时不仅要存权重，还要存：

```text
1. model weights
2. gradients
3. optimizer states
4. activations
```

以 Adam 为例，每个参数通常还会有一阶矩和二阶矩状态：

```text
m, v
```

所以 full fine-tuning 的真实显存压力远远不只是模型权重本身。

---

## 1.3 LoRA 的核心假设

LoRA 的直觉是：

> 大模型已经有很强的通用能力，下游任务不一定需要在完整高维参数空间里大幅修改，只需要在一个较低维的子空间里做任务适配。

也就是说，不直接学习完整的：

\[
\Delta W \in \mathbb{R}^{d_{out} \times d_{in}}
\]

而是学习一个低秩分解：

\[
\Delta W = BA
\]


In [ ]:

# 参数量计算：full fine-tuning 的单层线性层有多少参数？
def full_linear_params(d_in, d_out):
    return d_in * d_out

d_in = 4096
d_out = 4096

params_full = full_linear_params(d_in, d_out)
print(f"Full linear 参数量: {params_full:,}")



---

# 2️⃣ LoRA 核心公式：\(\Delta W = BA\)

原始线性层：

\[
y = Wx
\]

LoRA 冻结 \(W\)，额外训练两个小矩阵：

\[
A \in \mathbb{R}^{r \times d_{in}}
\]

\[
B \in \mathbb{R}^{d_{out} \times r}
\]

于是：

\[
\Delta W = BA
\]

\[
\Delta W \in \mathbb{R}^{d_{out} \times d_{in}}
\]

前向传播变成：

\[
y = Wx + \frac{\alpha}{r}BAx
\]

其中：

- \(W\)：冻结的原始权重
- \(A, B\)：可训练 LoRA 参数
- \(r\)：rank，控制低秩子空间大小
- \(\alpha\)：LoRA scaling 超参数
- \(\frac{\alpha}{r}\)：控制 LoRA 更新幅度

---

## 2.1 为什么参数少？

LoRA 参数量：

\[
\#A + \#B = r \cdot d_{in} + d_{out} \cdot r
\]

\[
= r(d_{in} + d_{out})
\]

如果 \(d_{in}=d_{out}=4096, r=8\)：

\[
\#LoRA = 8(4096 + 4096) = 65,536
\]

而 full linear 是：

\[
16,777,216
\]

差距约：

\[
\frac{16,777,216}{65,536}=256
\]

---

## 2.2 直觉类比

Full fine-tuning：

```text
重新装修整栋楼。
```

LoRA：

```text
不动主体结构，只加少量可替换的任务适配模块。
```

这也是为什么 LoRA adapter 可以很小、容易保存、容易切换。


In [ ]:

def lora_params(d_in, d_out, r):
    return r * d_in + d_out * r

ranks = [1, 2, 4, 8, 16, 32, 64, 128]
rows = []
for r in ranks:
    lp = lora_params(d_in, d_out, r)
    rows.append({
        "rank_r": r,
        "full_params": params_full,
        "lora_params": lp,
        "reduction_ratio": params_full / lp,
        "lora_percent_of_full": 100 * lp / params_full
    })

pd.DataFrame(rows)


In [ ]:

# 可视化 rank 对 LoRA 参数量的影响
df_rank = pd.DataFrame(rows)

plt.figure(figsize=(8, 4.8))
plt.plot(df_rank["rank_r"], df_rank["lora_params"], marker="o", label="LoRA params")
plt.axhline(params_full, linestyle="--", label="Full fine-tuning params")
plt.xscale("log", base=2)
plt.yscale("log")
plt.xlabel("LoRA Rank r")
plt.ylabel("Parameter Count (log scale)")
plt.title("LoRA rank 越大，参数量线性增长；但仍远小于 full fine-tuning")
plt.grid(alpha=0.3)
plt.legend()
plt.show()



---

# 3️⃣ LoRA 应该挂在哪些层？

在 Transformer 里，常见 target modules 包括：

```text
q_proj
k_proj
v_proj
o_proj
gate_proj
up_proj
down_proj
```

---

## 3.1 Attention 投影层

Attention 里有：

\[
Q = XW_Q
\]

\[
K = XW_K
\]

\[
V = XW_V
\]

\[
O = \text{Attention}(Q,K,V)W_O
\]

所以可以对：

```text
q_proj / k_proj / v_proj / o_proj
```

加 LoRA。

---

## 3.2 FFN / MLP 层

很多 LLM 的 FFN 不是简单两层，而是 gated MLP：

```text
gate_proj
up_proj
down_proj
```

也可以对这些层加 LoRA。

---

## 3.3 target_modules 的取舍

| 方案 | 优点 | 风险 |
|---|---|---|
| 只挂 q_proj/v_proj | 参数少，稳定，常见入门选择 | 表达能力有限 |
| q/k/v/o 全挂 | Attention 适配更强 | 参数增加 |
| attention + mlp 全挂 | 任务适配能力强 | 更易过拟合，显存更高 |
| rank 很高 | 容量更强 | 可能学坏基座风格 |

---

## 3.4 面试要点

> LoRA 不是只能用于 Attention。它本质上适用于线性层，只是在 Transformer 微调中常优先用于 Q/K/V/O 或 MLP 投影层，因为这些层对任务适配很关键。



---

# 4️⃣ PyTorch 实现：最小 LoRA Linear

我们现在实现一个最小版 LoRA Linear：

\[
y = xW^T + \frac{\alpha}{r}xA^TB^T
\]

注意 PyTorch 的 `nn.Linear` 习惯是：

```python
x @ weight.T
```

如果：

```text
weight: [d_out, d_in]
A: [r, d_in]
B: [d_out, r]
```

那么：

```text
x @ A.T        -> [batch, r]
(x @ A.T) @ B.T -> [batch, d_out]
```

---

## 4.1 初始化习惯

常见 LoRA 初始化：

```text
A 随机初始化
B 初始化为 0
```

这样训练一开始：

\[
BA = 0
\]

LoRA 不会立刻扰动原模型输出。


In [ ]:

class LoRALinear(nn.Module):
    def __init__(self, d_in, d_out, r=4, alpha=8, bias=False):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # 冻结的 base weight
        self.weight = nn.Parameter(torch.randn(d_out, d_in) / math.sqrt(d_in), requires_grad=False)

        # LoRA 参数：A 随机，B 为零
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, r))

        self.bias = nn.Parameter(torch.zeros(d_out), requires_grad=False) if bias else None

    def forward(self, x):
        base = F.linear(x, self.weight, self.bias)
        lora_update = (x @ self.A.T) @ self.B.T
        return base + self.scaling * lora_update

layer = LoRALinear(d_in=16, d_out=8, r=4, alpha=8)
x = torch.randn(5, 16)
y = layer(x)

print("输入形状:", tuple(x.shape))
print("输出形状:", tuple(y.shape))
print("可训练参数:")
for name, p in layer.named_parameters():
    print(name, tuple(p.shape), "trainable=", p.requires_grad)


In [ ]:

def count_params(module, trainable_only=False):
    total = 0
    for p in module.parameters():
        if (not trainable_only) or p.requires_grad:
            total += p.numel()
    return total

print("总参数量:", count_params(layer))
print("可训练参数量:", count_params(layer, trainable_only=True))



---

# 5️⃣ Toy Training：冻结基座，只训练 LoRA adapter

下面做一个小实验：

我们有一个真实目标线性映射：

\[
Y = XW_{target}^T
\]

但模型里冻结的 base weight 是一个有偏差的 \(W_{base}\)。

我们只训练 LoRA 的 A/B，让模型输出接近目标。

这对应真实微调里的直觉：

```text
base model 已经有通用能力
LoRA adapter 学任务差异
```


In [ ]:

torch.manual_seed(42)

# toy 数据
n = 512
d_in_toy = 20
d_out_toy = 10
r_toy = 4

X = torch.randn(n, d_in_toy)
W_target = torch.randn(d_out_toy, d_in_toy) / math.sqrt(d_in_toy)

Y = X @ W_target.T

# 构造 LoRA 模型，base weight 故意设得接近但不等于 target
model = LoRALinear(d_in_toy, d_out_toy, r=r_toy, alpha=8)
with torch.no_grad():
    model.weight.copy_(W_target + 0.25 * torch.randn_like(W_target))

optimizer = torch.optim.Adam([model.A, model.B], lr=0.05)

losses = []
for step in range(250):
    pred = model(X)
    loss = F.mse_loss(pred, Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print("最终 loss:", losses[-1])
print("可训练参数量:", count_params(model, trainable_only=True))
print("冻结 base 参数量:", model.weight.numel())


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.plot(losses)
plt.xlabel("Training Step")
plt.ylabel("MSE Loss")
plt.title("只训练 LoRA A/B，冻结 base weight：loss 下降曲线")
plt.grid(alpha=0.3)
plt.show()


In [ ]:

# 观察 LoRA 学到的 delta W 的秩
with torch.no_grad():
    delta_w = model.B @ model.A
    singular_values = torch.linalg.svdvals(delta_w).cpu().numpy()

plt.figure(figsize=(8, 4.8))
plt.bar(np.arange(len(singular_values)), singular_values)
plt.xlabel("Singular Value Index")
plt.ylabel("Singular Value")
plt.title("LoRA ΔW = BA 的奇异值：低秩更新直觉")
plt.show()

print("delta_w 形状:", tuple(delta_w.shape))
print("理论 rank 上限:", r_toy)
print("数值非零奇异值数量近似:", int((singular_values > 1e-6).sum()))



---

# 6️⃣ LoRA 的参数量 / 显存计算器

真实模型里，LoRA 不只挂一层，而是挂很多层、很多 target modules。

如果每个线性层都是：

\[
W \in \mathbb{R}^{d_{out} \times d_{in}}
\]

LoRA 参数量：

\[
r(d_{in}+d_{out})
\]

如果有 \(M\) 个 target modules：

\[
P_{LoRA,total} = M \cdot r(d_{in}+d_{out})
\]

---

## 6.1 估算例子

假设一个 7B 类模型：

- layers = 32
- 每层挂 4 个 attention projection：q/k/v/o
- 每个 projection 维度近似 4096 × 4096
- rank r = 8

那么 target modules 数量：

\[
M = 32 \times 4 = 128
\]

LoRA 参数量：

\[
128 \times 8(4096+4096)
\]

约 838 万参数。

这相比 7B 参数非常小。


In [ ]:

def estimate_lora_total_params(num_layers, modules_per_layer, d_in, d_out, r):
    return num_layers * modules_per_layer * r * (d_in + d_out)

estimate_rows = []
for r in [4, 8, 16, 32, 64]:
    p = estimate_lora_total_params(
        num_layers=32,
        modules_per_layer=4,
        d_in=4096,
        d_out=4096,
        r=r
    )
    estimate_rows.append({
        "rank": r,
        "LoRA params": p,
        "LoRA params (M)": p / 1e6,
        "percent of 7B": 100 * p / 7e9
    })

pd.DataFrame(estimate_rows)


In [ ]:

df_lora_est = pd.DataFrame(estimate_rows)

plt.figure(figsize=(8, 4.8))
plt.bar(df_lora_est["rank"].astype(str), df_lora_est["LoRA params (M)"])
for i, v in enumerate(df_lora_est["LoRA params (M)"]):
    plt.text(i, v + 0.2, f"{v:.1f}M", ha="center")
plt.xlabel("LoRA rank")
plt.ylabel("LoRA parameters (Million)")
plt.title("7B 类模型中，rank 对 LoRA adapter 参数量的影响")
plt.show()



---

# 7️⃣ QLoRA：4-bit 冻结基座 + LoRA adapter

QLoRA 可以理解成：

```text
Base model：4-bit 量化，冻结
LoRA adapter：正常训练
Gradient：反传经过量化基座，但只更新 LoRA
Optimizer states：只为 LoRA 参数维护
```

---

## 7.1 QLoRA 和 LoRA 的关系

LoRA：

```text
fp16/bf16 base model + train LoRA adapter
```

QLoRA：

```text
4-bit quantized base model + train LoRA adapter
```

所以 QLoRA 不是“训练 4-bit 权重”，而是：

> **冻结 4-bit 基座，把可训练部分限制在 LoRA adapter 上。**

---

## 7.2 QLoRA 的三个关键技术

### NF4

NF4 是一种适合近似正态分布权重的 4-bit 表示。大模型权重通常接近某种中心化分布，所以 NF4 比普通均匀 4-bit 更适合。

### Double Quantization

量化本身需要一些 scale / zero-point / quantization constants。Double quantization 继续压缩这些量化常数，进一步减少平均显存。

### Paged Optimizer

训练时 optimizer state 可能出现显存峰值。Paged optimizer 借鉴分页思想，把部分 optimizer 状态管理得更平滑，减少峰值 OOM 风险。

---

## 7.3 QLoRA 省什么，不省什么？

| 项目 | QLoRA 是否显著节省 |
|---|---|
| base model weights | 是，4-bit |
| full gradients | 是，不训练 base |
| optimizer states | 是，只针对 LoRA |
| LoRA adapter | 不大，本来就小 |
| activations | 不一定，仍受 batch 和 sequence length 影响 |
| KV Cache | 训练/推理时另算，不由 QLoRA 自动消失 |

---

## 7.4 面试一句话

> QLoRA 的核心不是直接训练 4-bit 权重，而是冻结 4-bit 量化基座，让梯度通过量化模型流向 LoRA adapter，只训练少量低秩参数。


In [ ]:

# Toy 4-bit quantization demo：用均匀量化模拟直觉
# 注意：这不是 NF4 的真实实现，只是帮助理解 4-bit 存储。
def quantize_uniform_4bit(w):
    w_min = w.min()
    w_max = w.max()
    scale = (w_max - w_min) / 15.0
    q = torch.clamp(torch.round((w - w_min) / scale), 0, 15).to(torch.uint8)
    return q, w_min, scale

def dequantize_uniform_4bit(q, w_min, scale):
    return q.float() * scale + w_min

W = torch.randn(1024, 1024)
q, w_min, scale = quantize_uniform_4bit(W)
W_hat = dequantize_uniform_4bit(q, w_min, scale)

mse = F.mse_loss(W_hat, W).item()
fp16_bytes = W.numel() * 2
int4_theoretical_bytes = W.numel() * 0.5

print("原始 W 形状:", tuple(W.shape))
print("量化重建 MSE:", mse)
print("fp16 理论存储:", fp16_bytes / 1024**2, "MB")
print("int4 理论存储:", int4_theoretical_bytes / 1024**2, "MB")
print("理论压缩倍率:", fp16_bytes / int4_theoretical_bytes)


In [ ]:

# 可视化量化前后的权重分布
sample = W.flatten()[:20000].numpy()
sample_hat = W_hat.flatten()[:20000].numpy()

plt.figure(figsize=(8, 4.8))
plt.hist(sample, bins=60, alpha=0.6, label="Original fp weight")
plt.hist(sample_hat, bins=60, alpha=0.6, label="4-bit reconstructed")
plt.xlabel("Weight value")
plt.ylabel("Frequency")
plt.title("Toy 4-bit uniform quantization：权重分布近似")
plt.legend()
plt.show()



---

# 8️⃣ Full FT / LoRA / QLoRA 显存直觉对比

为了建立工程直觉，我们做一个粗略估算。

假设：

- base model = 7B 参数
- fp16/bf16 = 2 bytes
- 4-bit = 0.5 bytes
- LoRA params = 8M
- Adam optimizer 对 trainable params 需要额外约 8 bytes/param（粗略示意）

---

## 8.1 注意

下面是教学估算，不是精确训练显存。

真实显存还包括：

```text
activation
temporary buffers
gradient checkpointing effect
KV cache
framework overhead
distributed training overhead
```

但这个估算足够帮你回答面试中的数量级问题。


In [ ]:

def gb(x):
    return x / (1024**3)

base_params = 7e9
lora_p = 8.4e6

# 粗略估算
full_ft = {
    "base weights fp16": base_params * 2,
    "gradients fp16": base_params * 2,
    "adam states rough": base_params * 8,
}

lora = {
    "frozen base fp16": base_params * 2,
    "lora weights fp16": lora_p * 2,
    "lora gradients fp16": lora_p * 2,
    "lora adam states rough": lora_p * 8,
}

qlora = {
    "frozen base 4bit": base_params * 0.5,
    "lora weights fp16": lora_p * 2,
    "lora gradients fp16": lora_p * 2,
    "lora adam states rough": lora_p * 8,
}

summary = []
for name, d in [("Full FT", full_ft), ("LoRA", lora), ("QLoRA", qlora)]:
    total = sum(d.values())
    summary.append({
        "method": name,
        "rough_train_memory_GB_excl_activations": gb(total),
        **{k: gb(v) for k, v in d.items()}
    })

pd.DataFrame(summary)


In [ ]:

mem_df = pd.DataFrame(summary)

plt.figure(figsize=(8, 4.8))
plt.bar(mem_df["method"], mem_df["rough_train_memory_GB_excl_activations"])
for i, v in enumerate(mem_df["rough_train_memory_GB_excl_activations"]):
    plt.text(i, v + 0.5, f"{v:.1f} GB", ha="center")
plt.ylabel("Rough Memory GB")
plt.title("Full FT vs LoRA vs QLoRA：粗略训练显存构成，不含 activations")
plt.show()



---

# 9️⃣ LoRA / QLoRA 常见排障

## 9.1 LoRA 微调后“学坏了”

常见原因：

```text
1. learning rate 太大
2. rank r 太高
3. target_modules 太多
4. 数据质量差
5. chat template 不匹配
6. EOS / BOS / assistant mask 错
7. 训练 epoch 太多，小数据过拟合
```

排查顺序：

```text
1. 固定 30-50 条 eval prompts
2. 对比 base vs LoRA
3. 降 learning rate
4. 降 rank
5. 减少 target modules
6. 检查 tokenizer / chat template / EOS
```

---

## 9.2 QLoRA 还是 OOM

常见原因：

```text
1. sequence length 太长
2. batch size 太大
3. gradient accumulation 配置不合理
4. 没开 gradient checkpointing
5. optimizer 错误地管理了 base model 参数
6. activation 显存仍然很大
7. Mac / CUDA / bitsandbytes 支持不匹配
```

记住：

> QLoRA 主要省 base weights 和 optimizer states，不会让 activation 免费消失。

---

## 9.3 部署 LoRA 时 tokenizer 崩 / 输出乱码

常见原因：

```text
1. 合并导出时 tokenizer 文件不匹配
2. chat template 不匹配
3. stop token 配错
4. base model 和 adapter 不对应
5. GGUF 导出时 special tokens 丢失
```

这和你之前遇到的部署问题直接相关：LoRA 权重本身训练成功，不代表合并、导出、模板和 tokenizer 都正确。



---

# 🔟 最小 RAG Chain：不依赖 API 的本地检索问答

现在进入 Day 7 的第二部分：LangChain / LangGraph 最小 RAG-Agent 闭环。

为了不让你被 API key、模型服务、网络问题卡住，我们先用纯 Python 写一个“玩具 RAG”。

结构是：

```text
user question
    ↓
retriever 检索本地文档
    ↓
rerank 简单排序
    ↓
answer chain 生成答案
    ↓
citation / source 返回
    ↓
trace 记录
```

---

## 10.1 为什么 RAG 不是模型内部记忆？

Retriever 检索的是外部知识库：

```text
documents / vector store / database / files
```

它不是模型参数里的知识。

模型回答时只是把检索结果作为上下文输入：

```text
prompt = question + retrieved_context
```

所以 RAG 的关键不是“让模型记住”，而是：

> **在回答前动态取回相关上下文，并约束回答必须基于上下文。**


In [ ]:

# 一个很小的本地知识库
documents = [
    {
        "id": "doc_attention",
        "title": "Scaled Dot-Product Attention",
        "text": "Attention uses QK^T divided by sqrt(d_k), followed by softmax, then weighted sum of V."
    },
    {
        "id": "doc_kv_cache",
        "title": "KV Cache",
        "text": "KV Cache stores historical keys and values during autoregressive decoding. It reduces repeated computation but grows linearly with context length."
    },
    {
        "id": "doc_rope",
        "title": "RoPE",
        "text": "Rotary Position Embedding applies position-dependent rotations to Q and K, enabling relative position awareness in attention scores."
    },
    {
        "id": "doc_lora",
        "title": "LoRA",
        "text": "LoRA freezes base weights and trains low-rank matrices A and B so that delta W equals B times A."
    },
    {
        "id": "doc_qlora",
        "title": "QLoRA",
        "text": "QLoRA uses a frozen 4-bit quantized base model and trains LoRA adapters, reducing memory while preserving finetuning quality."
    },
]

documents


In [ ]:

# 一个极简 bag-of-words 检索器
import re
from collections import Counter

def tokenize(text):
    return re.findall(r"[a-zA-Z0-9_]+", text.lower())

def build_vocab(docs):
    vocab = {}
    for doc in docs:
        for tok in tokenize(doc["title"] + " " + doc["text"]):
            if tok not in vocab:
                vocab[tok] = len(vocab)
    return vocab

def vectorize(text, vocab):
    vec = np.zeros(len(vocab))
    counts = Counter(tokenize(text))
    for tok, c in counts.items():
        if tok in vocab:
            vec[vocab[tok]] = c
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

def cosine(a, b):
    return float(np.dot(a, b))

vocab = build_vocab(documents)
doc_vectors = np.stack([vectorize(doc["title"] + " " + doc["text"], vocab) for doc in documents])

def retrieve(query, top_k=3):
    qv = vectorize(query, vocab)
    scores = [cosine(qv, dv) for dv in doc_vectors]
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

query = "Why does QLoRA save memory compared with full finetuning?"
retrieved = retrieve(query)
pd.DataFrame([
    {"doc_id": d["id"], "title": d["title"], "score": round(s, 3), "text": d["text"]}
    for d, s in retrieved
])


In [ ]:

def answer_with_context(question, retrieved_docs):
    # 这里不用 LLM，做一个可控的模板回答，重点是 RAG 结构。
    context_lines = []
    citations = []
    for doc, score in retrieved_docs:
        if score > 0:
            context_lines.append(f"- {doc['text']}")
            citations.append(doc["id"])

    if not context_lines:
        return {
            "answer": "我没有在本地知识库中找到足够相关的上下文。",
            "citations": []
        }

    answer = (
        f"问题：{question}\n\n"
        f"基于检索到的上下文，可以这样回答：\n"
        + "\n".join(context_lines)
        + "\n\n"
        f"引用来源：{', '.join(citations)}"
    )
    return {"answer": answer, "citations": citations}

result = answer_with_context(query, retrieved)
print(result["answer"])



---

# 1️⃣1️⃣ 最小 LangGraph 风格 StateGraph

LangGraph 的关键直觉是：

> 节点不直接互相传乱七八糟的参数，而是读写一个共享 state。

一个节点签名可以理解为：

```python
State -> Partial[State]
```

例如：

```text
retrieve_context(state) -> {"retrieved_docs": ...}
answer_with_context(state) -> {"answer": ...}
verify_answer(state) -> {"verification": ...}
```

下面我们不用安装 LangGraph，先手写一个最小状态机。

---

## 11.1 为什么要用状态机？

普通 chain：

```text
A → B → C
```

适合固定流程。

复杂 agent：

```text
A → B → C
    ↑   ↓
    D ← E
```

会有循环、条件分支、失败回退、人工介入。

这时用 state graph 更清晰：

```text
state 记录所有中间结果
node 只负责读写 state
edge 控制下一步去哪
trace 记录全流程
```


In [ ]:

def node_retrieve_context(state):
    start = time.time()
    retrieved = retrieve(state["question"], top_k=state.get("top_k", 3))
    latency = time.time() - start

    return {
        "retrieved_docs": retrieved,
        "trace": state.get("trace", []) + [{
            "node": "retrieve_context",
            "latency_ms": round(latency * 1000, 3),
            "docs": [{"id": d["id"], "score": round(s, 3)} for d, s in retrieved]
        }]
    }

def node_answer_with_context(state):
    start = time.time()
    ans = answer_with_context(state["question"], state["retrieved_docs"])
    latency = time.time() - start

    return {
        "answer": ans["answer"],
        "citations": ans["citations"],
        "trace": state.get("trace", []) + [{
            "node": "answer_with_context",
            "latency_ms": round(latency * 1000, 3),
            "citation_count": len(ans["citations"])
        }]
    }

def node_verify_answer(state):
    start = time.time()
    ok = len(state.get("citations", [])) > 0 and "基于检索到的上下文" in state.get("answer", "")
    latency = time.time() - start

    return {
        "verification": {
            "grounded": ok,
            "reason": "answer has citations and uses retrieved context" if ok else "answer lacks citations or context"
        },
        "trace": state.get("trace", []) + [{
            "node": "verify_answer",
            "latency_ms": round(latency * 1000, 3),
            "grounded": ok
        }]
    }

def run_minimal_graph(question):
    state = {
        "question": question,
        "top_k": 3,
        "trace": []
    }

    for node in [node_retrieve_context, node_answer_with_context, node_verify_answer]:
        update = node(state)
        state.update(update)

    return state

state = run_minimal_graph("Explain LoRA and QLoRA memory saving.")
print(state["answer"])
print("\nVerification:", state["verification"])
print("\nTrace:")
pd.DataFrame(state["trace"])


In [ ]:

# 可视化最小 StateGraph
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.axis("off")

nodes = [
    ("START", 0.05, 0.5),
    ("retrieve_context", 0.28, 0.5),
    ("answer_with_context", 0.55, 0.5),
    ("verify_answer", 0.80, 0.5),
    ("END", 0.97, 0.5),
]

for label, x, y in nodes:
    ax.text(x, y, label, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black"))

for i in range(len(nodes)-1):
    x1, y1 = nodes[i][1], nodes[i][2]
    x2, y2 = nodes[i+1][1], nodes[i+1][2]
    ax.annotate("", xy=(x2-0.06, y2), xytext=(x1+0.06, y1),
                arrowprops=dict(arrowstyle="->", lw=1.5))

ax.set_title("最小 LangGraph 风格 RAG-Agent 状态机")
plt.show()



---

# 1️⃣2️⃣ LangChain / LangGraph 工程映射

## 12.1 LangChain 更像组件库

LangChain 适合快速组合：

```text
PromptTemplate
ChatModel
Retriever
Tool
OutputParser
Runnable
```

LCEL / Runnable 的直觉是：

```text
input | prompt | model | parser
```

也就是把组件串成可调用链。

---

## 12.2 LangGraph 更像状态机运行时

LangGraph 适合：

```text
多节点
条件分支
循环
checkpoint
human-in-the-loop
长任务恢复
多 agent 协作
```

核心是：

```text
State
Node
Edge
Checkpoint
```

---

## 12.3 普通 RAG vs Agentic RAG

| 类型 | 流程 | 优点 | 风险 |
|---|---|---|---|
| 普通 RAG | 固定 检索→回答 | 稳定、低延迟、好评估 | 不够灵活 |
| Agentic RAG | 模型决定是否检索、是否多轮工具调用 | 灵活，可处理复杂问题 | 工具循环、延迟高、可控性差 |

---

## 12.4 生产 trace 至少记录什么？

```text
request_id
run_id
question
rewritten_query
retrieved_docs
retrieval_scores
rerank_scores
tool_calls
node_latency
llm_latency
token_usage
final_answer
citations
verification_result
error
user_feedback
```

没有 trace 的 RAG / Agent 基本没法排障。


In [ ]:

# 把一次运行的 trace 保存成 jsonl 格式示意
trace_record = {
    "request_id": "req_demo_001",
    "question": state["question"],
    "citations": state["citations"],
    "verification": state["verification"],
    "trace": state["trace"],
}

print(json.dumps(trace_record, ensure_ascii=False, indent=2))



---

# 1️⃣3️⃣ Day 7 面试高频问答

## Q1：LoRA 为什么能减少可训练参数？

**满分回答：**

LoRA 冻结原始权重 \(W\)，不直接训练完整的 \(\Delta W\)，而是用两个低秩矩阵表示更新：

\[
\Delta W = BA
\]

其中：

\[
A \in \mathbb{R}^{r \times d_{in}}, \quad B \in \mathbb{R}^{d_{out} \times r}
\]

可训练参数从：

\[
d_{out}d_{in}
\]

降为：

\[
r(d_{in}+d_{out})
\]

当 \(r\) 很小时，参数量大幅下降。

---

## Q2：LoRA rank 太小或太大会怎样？

**满分回答：**

rank 太小，低秩子空间表达能力不足，可能欠拟合；rank 太大，参数量和显存增加，也更容易在小数据上过拟合或破坏基座模型能力。rank 是适配能力和稳定性之间的 trade-off。

---

## Q3：QLoRA 和 LoRA 的区别是什么？

**满分回答：**

LoRA 通常在 fp16/bf16 基座上训练低秩 adapter；QLoRA 将基座模型量化为 4-bit 并冻结，只训练 LoRA adapter。QLoRA 通过 4-bit base weights、NF4、double quantization 和 paged optimizers 降低显存，但 activation 仍然受 sequence length 和 batch size 影响。

---

## Q4：QLoRA 是不是直接训练 4-bit 权重？

**满分回答：**

不是。QLoRA 冻结 4-bit 量化基座，让梯度通过量化权重反传到 LoRA adapter，只更新 LoRA 参数，而不是直接更新 4-bit base weights。

---

## Q5：LoRA 微调后模型变笨了，怎么排查？

**满分回答：**

先固定评测集比较 base 和 adapter，然后检查 learning rate、rank、target_modules、epoch、数据质量、chat template、EOS/BOS、assistant mask。如果任务增强但通用能力下降，通常降低 learning rate、rank 或训练轮数。

---

## Q6：普通 RAG Chain 和 Agentic RAG 的区别？

**满分回答：**

普通 RAG Chain 固定执行检索、拼上下文、回答，稳定低延迟；Agentic RAG 让模型或策略决定是否检索、检索什么、是否二次检索或调用工具，更灵活但延迟和可控性更差，也更需要 trace 和评估。

---

## Q7：LangChain 和 LangGraph 的区别？

**满分回答：**

LangChain 更像应用组件库，适合组合 prompt、model、retriever、tool 和 parser；LangGraph 更像状态机和 agent runtime，适合复杂多节点、循环、条件分支、checkpoint、human-in-the-loop 和长任务恢复。

---

## Q8：RAG 评估看什么？

**满分回答：**

检索侧看 Recall@K、MRR、Context Precision；生成侧看 Faithfulness、Answer Correctness、Citation Accuracy；线上还要看延迟、失败率、无答案率、用户反馈和工具调用错误率。



---

# 1️⃣4️⃣ 今日最小作业

## 作业 1：改 LoRA rank

把 toy training 里的：

```python
r_toy = 4
```

改成：

```python
r_toy = 1
r_toy = 2
r_toy = 8
```

观察 loss 曲线变化，并回答：

```text
rank 越大一定越好吗？
```

---

## 作业 2：改 LoRA alpha

把：

```python
alpha = 8
```

改成：

```python
alpha = 1
alpha = 16
alpha = 32
```

观察训练稳定性和最终 loss。

---

## 作业 3：写 QLoRA 显存解释

用自己的话写 5 行：

```text
QLoRA 省在哪里？
QLoRA 不省哪里？
为什么 activation 仍然可能 OOM？
```

---

## 作业 4：扩展最小 RAG

往 `documents` 里新增 3 条文档：

```text
PagedAttention
MoE
RoPE
```

然后问一个跨主题问题：

```text
为什么 GQA 和 PagedAttention 都能改善长上下文服务？
```

观察 retriever 是否能召回正确文档。

---

## 作业 5：扩展 StateGraph

加入一个条件分支：

```text
如果 verification.grounded == False
    → fallback_answer
否则
    → END
```

这就是 LangGraph 的核心价值：不是只会线性 chain，而是能做状态驱动的流程控制。



---

# 1️⃣5️⃣ 推荐学习平台 / 视频 / GitHub（3–5 个高质量资源）

## 资源 1：LoRA 原论文
- 类型：论文
- 用途：理解 LoRA 的低秩更新、冻结基座和参数效率。
- 链接：https://arxiv.org/abs/2106.09685

## 资源 2：QLoRA 原论文
- 类型：论文
- 用途：理解 NF4、double quantization、paged optimizer，以及 4-bit 冻结基座 + LoRA adapter。
- 链接：https://arxiv.org/abs/2305.14314

## 资源 3：Microsoft LoRA GitHub
- 类型：GitHub 仓库
- 用途：看官方 loralib 的实现方式和 PyTorch 集成示例。
- 链接：https://github.com/microsoft/LoRA

## 资源 4：LoRA Explained With Code & Analogy
- 类型：YouTube 视频
- 用途：用代码和类比理解 LoRA。
- 链接：https://www.youtube.com/watch?v=IndUu-lkJFU

## 资源 5：LangGraph 官方文档 / Academy
- 类型：官方文档 / 课程
- 用途：学习 StateGraph、checkpoint、agent orchestration。
- 链接：https://docs.langchain.com/oss/python/langgraph/graph-api

---

# ✅ 今日一句话总总结

> **LoRA 用低秩矩阵学习任务增量，QLoRA 用 4-bit 冻结基座进一步压低显存；而 RAG-Agent 闭环则把底层模型能力接到工程应用中：检索、回答、验证、trace 和状态机编排共同决定生产可用性。**
